In [4]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps

from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

print("Python:", sys.version.split()[0])
print("PySINDy:", ps.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("IMPORTACIONES CORRECTAS")

Python: 3.11.17
PySINDy: 2.1.0
NumPy: 2.4.6
Pandas: 2.3.3
IMPORTACIONES CORRECTAS


In [5]:
# Reconstrucción puntual del segmento continuo faltante de RRab

rrab_source_path = (
    PROJECT_ROOT
    / "07_data"
    / "processed"
    / "pilot_quality_zero"
    / "rrab_pilot.csv"
)

rrab_output_path = (
    SEGMENT_DIR
    / "rrab_continuous_segment.csv"
)

print("Fuente existente:", rrab_source_path.exists())
print("Ruta de salida:", rrab_output_path)

if not rrab_source_path.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo fuente: {rrab_source_path}"
    )

rrab_table = pd.read_csv(rrab_source_path)

required_columns = {
    "time_btjd",
    "sap_normalized",
}

missing_columns = required_columns - set(rrab_table.columns)

if missing_columns:
    raise ValueError(
        f"Faltan columnas en RRab: {missing_columns}"
    )

# Conservar solo tiempo y SAP normalizado, con valores válidos
rrab_table = (
    rrab_table[
        ["time_btjd", "sap_normalized"]
    ]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sort_values("time_btjd")
    .drop_duplicates("time_btjd")
    .reset_index(drop=True)
)

rrab_time = rrab_table["time_btjd"].to_numpy(
    dtype=float
)

time_differences = np.diff(rrab_time)

positive_differences = time_differences[
    time_differences > 0
]

if len(positive_differences) == 0:
    raise RuntimeError(
        "No se pudieron calcular intervalos positivos."
    )

median_cadence_days = np.median(
    positive_differences
)

gap_factor = 5.0

gap_indices = np.flatnonzero(
    time_differences
    > gap_factor * median_cadence_days
)

segment_boundaries = np.concatenate(
    [
        [0],
        gap_indices + 1,
        [len(rrab_table)],
    ]
)

candidate_segments = []

for segment_id, (start, end) in enumerate(
    zip(
        segment_boundaries[:-1],
        segment_boundaries[1:],
    )
):
    segment = rrab_table.iloc[
        start:end
    ].copy()

    if len(segment) < 2:
        continue

    duration_days = (
        segment["time_btjd"].iloc[-1]
        - segment["time_btjd"].iloc[0]
    )

    approximate_cycles = (
        duration_days / 0.56695323
    )

    candidate_segments.append(
        {
            "segment_id": segment_id,
            "start": int(start),
            "end": int(end),
            "n_points": len(segment),
            "duration_days": float(duration_days),
            "approximate_cycles": float(
                approximate_cycles
            ),
        }
    )

rrab_segment_inventory = pd.DataFrame(
    candidate_segments
)

print("\nSegmentos RRab encontrados:")
print(
    rrab_segment_inventory.to_string(
        index=False
    )
)

Fuente existente: True
Ruta de salida: C:\Users\Maor\Documents\TESS_Dynamical_Equations\07_data\processed\pilot_continuous_segments\rrab_continuous_segment.csv

Segmentos RRab encontrados:
 segment_id  start   end  n_points  duration_days  approximate_cycles
          0      0  3225      3225       4.543072            8.013134
          1   3225  3234         9       0.011111            0.019598
          2   3234  3484       250       0.366668            0.646734
          3   3484  6370      2886       4.097233            7.226756
          4   6370  9189      2819       3.980567            7.020980
          5   9189 12214      3025       4.281916            7.552503
          6  12214 15223      3009       4.238856            7.476553
          7  15223 18292      3069       4.316634            7.613739


In [7]:
# Selección revisada del segmento continuo de RRab
# El criterio de 10 ciclos fue demasiado restrictivo.

minimum_rrab_points = 500
minimum_rrab_cycles = 3.0

print("Inventario disponible de RRab:")
print(
    rrab_segment_inventory[
        [
            "segment_id",
            "n_points",
            "duration_days",
            "approximate_cycles",
        ]
    ]
    .sort_values(
        "duration_days",
        ascending=False,
    )
    .to_string(index=False)
)

eligible_rrab_segments = (
    rrab_segment_inventory[
        (
            rrab_segment_inventory["n_points"]
            >= minimum_rrab_points
        )
        & (
            rrab_segment_inventory["approximate_cycles"]
            >= minimum_rrab_cycles
        )
    ]
    .copy()
)

if eligible_rrab_segments.empty:
    # Plan de respaldo: escoger el segmento más largo
    # siempre que tenga al menos 500 puntos.
    eligible_rrab_segments = (
        rrab_segment_inventory[
            rrab_segment_inventory["n_points"]
            >= minimum_rrab_points
        ]
        .copy()
    )

if eligible_rrab_segments.empty:
    raise RuntimeError(
        "RRab no contiene ningún segmento continuo "
        "con al menos 500 puntos."
    )

selected_rrab_segment_info = (
    eligible_rrab_segments
    .sort_values(
        ["duration_days", "n_points"],
        ascending=[False, False],
    )
    .iloc[0]
)

selected_start = int(
    selected_rrab_segment_info["start"]
)

selected_end = int(
    selected_rrab_segment_info["end"]
)

rrab_selected_segment = (
    rrab_table
    .iloc[selected_start:selected_end]
    .copy()
)

rrab_selected_segment[
    "time_relative_days"
] = (
    rrab_selected_segment["time_btjd"]
    - rrab_selected_segment["time_btjd"].iloc[0]
)

rrab_selected_segment = (
    rrab_selected_segment[
        [
            "time_btjd",
            "time_relative_days",
            "sap_normalized",
        ]
    ]
    .reset_index(drop=True)
)

rrab_selected_segment.to_csv(
    rrab_output_path,
    index=False,
)

print("\nSegmento RRab seleccionado:")
print(
    selected_rrab_segment_info[
        [
            "segment_id",
            "n_points",
            "duration_days",
            "approximate_cycles",
        ]
    ].to_string()
)

print(
    "\nArchivo RRab guardado:",
    rrab_output_path.exists(),
)

print(
    "Puntos guardados:",
    len(rrab_selected_segment),
)

print(
    "Duración guardada:",
    f"{rrab_selected_segment['time_relative_days'].iloc[-1]:.6f}",
    "días",
)

print(
    "Ciclos aproximados:",
    f"{selected_rrab_segment_info['approximate_cycles']:.3f}",
)

Inventario disponible de RRab:
 segment_id  n_points  duration_days  approximate_cycles
          0      3225       4.543072            8.013134
          7      3069       4.316634            7.613739
          5      3025       4.281916            7.552503
          6      3009       4.238856            7.476553
          3      2886       4.097233            7.226756
          4      2819       3.980567            7.020980
          2       250       0.366668            0.646734
          1         9       0.011111            0.019598

Segmento RRab seleccionado:
segment_id               0.000000
n_points              3225.000000
duration_days            4.543072
approximate_cycles       8.013134

Archivo RRab guardado: True
Puntos guardados: 3225
Duración guardada: 4.543072 días
Ciclos aproximados: 8.013


In [9]:
rrab_saved_check = pd.read_csv(
    rrab_output_path
)

rrab_time_check = (
    rrab_saved_check[
        "time_relative_days"
    ]
    .to_numpy(dtype=float)
)

rrab_flux_check = (
    rrab_saved_check[
        "sap_normalized"
    ]
    .to_numpy(dtype=float)
)

rrab_dt_check = np.diff(
    rrab_time_check
)

median_dt_seconds = float(
    np.median(rrab_dt_check) * 86400.0
)

largest_dt_seconds = float(
    np.max(rrab_dt_check) * 86400.0
)

interval_ratio = (
    largest_dt_seconds
    / median_dt_seconds
)

print("Filas:", len(rrab_saved_check))

print(
    "Cadencia mediana:",
    f"{median_dt_seconds:.6f} segundos",
)

print(
    "Intervalo máximo:",
    f"{largest_dt_seconds:.6f} segundos",
)

print(
    "Razón máximo/mediana:",
    f"{interval_ratio:.3f}",
)

print(
    "Valores de flujo finitos:",
    bool(np.all(np.isfinite(rrab_flux_check))),
)

assert len(rrab_saved_check) >= 500
assert np.all(np.diff(rrab_time_check) > 0)
assert np.all(np.isfinite(rrab_flux_check))
assert interval_ratio <= 5.01

print("SEGMENTO RRab VALIDADO")

Filas: 3225
Cadencia mediana: 120.000432 segundos
Intervalo máximo: 480.002189 segundos
Razón máximo/mediana: 4.000
Valores de flujo finitos: True
SEGMENTO RRab VALIDADO


In [8]:
PROJECT_ROOT = Path.cwd().parent

SEGMENT_DIR = (
    PROJECT_ROOT
    / "07_data"
    / "processed"
    / "pilot_continuous_segments"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "08_results"
)

MODEL_DIR = RESULTS_DIR / "models"
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"

for directory in [
    MODEL_DIR,
    TABLE_DIR,
    FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Proyecto:", PROJECT_ROOT)
print("Segmentos:", SEGMENT_DIR)
print("Directorio existente:", SEGMENT_DIR.exists())

segment_files = sorted(
    SEGMENT_DIR.glob(
        "*_continuous_segment.csv"
    )
)

print("\nArchivos encontrados:")

for file_path in segment_files:
    print("-", file_path.name)

if len(segment_files) != 3:
    raise RuntimeError(
        "Se esperaban exactamente tres segmentos CSV. "
        f"Se encontraron {len(segment_files)}."
    )

print("\nTRES SEGMENTOS DISPONIBLES")

Proyecto: C:\Users\Maor\Documents\TESS_Dynamical_Equations
Segmentos: C:\Users\Maor\Documents\TESS_Dynamical_Equations\07_data\processed\pilot_continuous_segments
Directorio existente: True

Archivos encontrados:
- dsct_continuous_segment.csv
- ew_continuous_segment.csv
- rrab_continuous_segment.csv

TRES SEGMENTOS DISPONIBLES


In [16]:
periods_days = {
    "RRab": 0.56695323,
    "DSCT": 0.19381569,
    "EW": 0.33365223,
}

file_mapping = {
    "RRab": SEGMENT_DIR / "rrab_continuous_segment.csv",
    "DSCT": SEGMENT_DIR / "dsct_continuous_segment.csv",
    "EW": SEGMENT_DIR / "ew_continuous_segment.csv",
}

raw_trajectories = {}

for variability_class, file_path in file_mapping.items():
    if not file_path.exists():
        raise FileNotFoundError(
            f"No existe el archivo: {file_path}"
        )

    table = pd.read_csv(file_path)

    required_columns = {
        "time_relative_days",
        "sap_normalized",
    }

    missing_columns = (
        required_columns - set(table.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{variability_class}: "
            f"faltan columnas {missing_columns}"
        )

    table = (
        table[
            [
                "time_relative_days",
                "sap_normalized",
            ]
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values("time_relative_days")
        .drop_duplicates("time_relative_days")
        .reset_index(drop=True)
    )

    raw_trajectories[
        variability_class
    ] = table

    duration_days = float(
        table["time_relative_days"].iloc[-1]
        - table["time_relative_days"].iloc[0]
    )

    approximate_cycles = (
        duration_days
        / periods_days[variability_class]
    )

    print(
        f"{variability_class}: "
        f"{len(table)} puntos, "
        f"{duration_days:.5f} días, "
        f"{approximate_cycles:.2f} ciclos"
    )

RRab: 3225 puntos, 4.54307 días, 8.01 ciclos
DSCT: 4635 puntos, 6.43608 días, 33.21 ciclos
EW: 4588 puntos, 6.37374 días, 19.10 ciclos


In [22]:
def prepare_sindy_trajectory(
    table,
    period_days,
    nominal_cadence_seconds=120.0,
):
    time_days = table[
        "time_relative_days"
    ].to_numpy(dtype=float)

    flux = table[
        "sap_normalized"
    ].to_numpy(dtype=float)

    if len(time_days) < 20:
        raise ValueError(
            "La trayectoria contiene muy pocos puntos."
        )

    if not np.all(np.diff(time_days) > 0):
        raise ValueError(
            "Los tiempos deben ser estrictamente crecientes."
        )

    cadence_days = (
        nominal_cadence_seconds / 86400.0
    )

    # Construir una malla uniforme sin superar
    # el último instante observado
    time_span_days = (
        time_days[-1] - time_days[0]
    )

    number_of_intervals = int(
        np.floor(
            time_span_days / cadence_days
        )
    )

    uniform_time_days = (
        time_days[0]
        + np.arange(
            number_of_intervals + 1,
            dtype=float,
        )
        * cadence_days
    )

    uniform_time_days = uniform_time_days[
        uniform_time_days <= time_days[-1]
    ]

    interpolator = interp1d(
        time_days,
        flux,
        kind="linear",
        bounds_error=True,
    )

    uniform_flux = interpolator(
        uniform_time_days
    )

    # Tiempo adimensional en ciclos
    time_cycles = (
        uniform_time_days
        - uniform_time_days[0]
    ) / period_days

    delta_cycles = float(
        np.median(
            np.diff(time_cycles)
        )
    )

    # Centrar y escalar el flujo
    flux_mean = float(
        np.mean(uniform_flux)
    )

    flux_scale = float(
        np.std(uniform_flux)
    )

    if not np.isfinite(flux_scale):
        raise ValueError(
            "La escala del flujo no es finita."
        )

    if flux_scale <= 0:
        raise ValueError(
            "La escala del flujo debe ser positiva."
        )

    scaled_flux = (
        uniform_flux - flux_mean
    ) / flux_scale

    samples_per_cycle = (
        period_days / cadence_days
    )

    # Ventana próxima al 5 % de un ciclo
    window_length = int(
        np.round(
            0.05 * samples_per_cycle
        )
    )

    window_length = max(
        window_length,
        7,
    )

    if window_length % 2 == 0:
        window_length += 1

    maximum_window = (
        len(scaled_flux) - 1
    )

    if maximum_window % 2 == 0:
        maximum_window -= 1

    window_length = min(
        window_length,
        maximum_window,
    )

    if window_length < 5:
        raise ValueError(
            "Ventana Savitzky-Golay insuficiente."
        )

    smoothed_flux = savgol_filter(
        scaled_flux,
        window_length=window_length,
        polyorder=3,
        deriv=0,
        delta=delta_cycles,
        mode="interp",
    )

    first_derivative = savgol_filter(
        scaled_flux,
        window_length=window_length,
        polyorder=3,
        deriv=1,
        delta=delta_cycles,
        mode="interp",
    )

    second_derivative = savgol_filter(
        scaled_flux,
        window_length=window_length,
        polyorder=3,
        deriv=2,
        delta=delta_cycles,
        mode="interp",
    )

    # Retirar los bordes afectados por el filtro
    edge_trim = window_length

    if 2 * edge_trim >= len(smoothed_flux):
        raise ValueError(
            "La ventana de suavizado es demasiado grande "
            "para la trayectoria."
        )

    valid_slice = slice(
        edge_trim,
        len(smoothed_flux) - edge_trim,
    )

    trimmed_time_cycles = (
        time_cycles[valid_slice]
    )

    trimmed_raw_flux = (
        scaled_flux[valid_slice]
    )

    trimmed_flux = (
        smoothed_flux[valid_slice]
    )

    trimmed_first_derivative = (
        first_derivative[valid_slice]
    )

    trimmed_second_derivative = (
        second_derivative[valid_slice]
    )

    state_matrix = np.column_stack(
        [
            trimmed_flux,
            trimmed_first_derivative,
        ]
    )

    derivative_matrix = np.column_stack(
        [
            trimmed_first_derivative,
            trimmed_second_derivative,
        ]
    )

    return {
        "time_cycles": trimmed_time_cycles,
        "delta_cycles": delta_cycles,
        "raw_scaled_flux": trimmed_raw_flux,
        "smoothed_flux": trimmed_flux,
        "first_derivative": (
            trimmed_first_derivative
        ),
        "second_derivative": (
            trimmed_second_derivative
        ),
        "state_matrix": state_matrix,
        "derivative_matrix": derivative_matrix,
        "window_length": window_length,
        "samples_per_cycle": samples_per_cycle,
        "original_points": len(time_days),
        "uniform_points": len(
            uniform_time_days
        ),
        "retained_points": len(
            trimmed_time_cycles
        ),
        "interpolated_points": int(
            len(uniform_time_days)
            - len(time_days)
        ),
        "uniform_start_days": float(
            uniform_time_days[0]
        ),
        "uniform_end_days": float(
            uniform_time_days[-1]
        ),
        "observed_end_days": float(
            time_days[-1]
        ),
    }

In [26]:
sindy_trajectories = {}

preparation_rows = []

for variability_class, table in (
    raw_trajectories.items()
):
    trajectory = prepare_sindy_trajectory(
        table=table,
        period_days=periods_days[
            variability_class
        ],
    )

    sindy_trajectories[
        variability_class
    ] = trajectory

    preparation_rows.append(
        {
            "variability_class": (
                variability_class
            ),
            "original_points": trajectory[
                "original_points"
            ],
            "uniform_points": trajectory[
                "uniform_points"
            ],
            "interpolated_points": trajectory[
                "interpolated_points"
            ],
            "retained_points": trajectory[
                "retained_points"
            ],
            "samples_per_cycle": trajectory[
                "samples_per_cycle"
            ],
            "savgol_window": trajectory[
                "window_length"
            ],
            "delta_cycles": trajectory[
                "delta_cycles"
            ],
        }
    )

preparation_summary = pd.DataFrame(
    preparation_rows
)

print(
    preparation_summary.to_string(
        index=False
    )
)

variability_class  original_points  uniform_points  interpolated_points  retained_points  samples_per_cycle  savgol_window  delta_cycles
             RRab             3225            3272                   47             3230         408.206326             21      0.002450
             DSCT             4635            4634                   -1             4620         139.547297              7      0.007166
               EW             4588            4590                    2             4564         240.229606             13      0.004163


In [28]:
for variability_class, trajectory in sindy_trajectories.items():
    assert (
        trajectory["uniform_end_days"]
        <= trajectory["observed_end_days"]
    )

    assert np.all(
        np.isfinite(trajectory["state_matrix"])
    )

    assert np.all(
        np.isfinite(trajectory["derivative_matrix"])
    )

    # Calculamos el margen fuera del f-string para evitar el SyntaxError
    margen_segundos = (trajectory['observed_end_days'] - trajectory['uniform_end_days']) * 86400.0

    print(
        variability_class,
        "| margen final:",
        f"{margen_segundos:.6f} segundos",
        "| interpolados:",
        trajectory["interpolated_points"],
    )

print("PREPARACIÓN PARA SINDy COMPLETADA")


RRab | margen final: 1.453898 segundos | interpolados: 47
DSCT | margen final: 116.971075 segundos | interpolados: -1
EW | margen final: 11.466410 segundos | interpolados: 2
PREPARACIÓN PARA SINDy COMPLETADA


In [29]:
derivative_rows = []

for variability_class, trajectory in sindy_trajectories.items():
    x = trajectory["smoothed_flux"]
    velocity = trajectory["first_derivative"]
    acceleration = trajectory["second_derivative"]

    derivative_rows.append(
        {
            "variability_class": variability_class,
            "n_points": len(x),
            "x_min": float(np.min(x)),
            "x_max": float(np.max(x)),
            "velocity_std": float(np.std(velocity)),
            "acceleration_std": float(np.std(acceleration)),
            "all_finite": bool(
                np.all(np.isfinite(x))
                and np.all(np.isfinite(velocity))
                and np.all(np.isfinite(acceleration))
            ),
        }
    )

derivative_summary = pd.DataFrame(derivative_rows)

print(derivative_summary.to_string(index=False))

assert derivative_summary["all_finite"].all()

print("DERIVADAS LISTAS PARA SINDy")

variability_class  n_points     x_min    x_max  velocity_std  acceleration_std  all_finite
             RRab      3230 -1.483611 2.123533     11.979726        346.336704        True
             DSCT      4620 -1.693199 1.933658      6.589260         52.598989        True
               EW      4564 -1.928156 1.182116     14.205887        263.937377        True
DERIVADAS LISTAS PARA SINDy


In [30]:
def fit_initial_sindy_model(
    trajectory,
    threshold=0.05,
):
    feature_library = ps.PolynomialLibrary(
        degree=3,
        include_bias=True,
    )

    optimizer = ps.STLSQ(
        threshold=threshold,
        alpha=1e-5,
        max_iter=100,
        normalize_columns=True,
    )

    model = ps.SINDy(
        feature_library=feature_library,
        optimizer=optimizer,
    )

    model.fit(
        trajectory["state_matrix"],
        t=trajectory["delta_cycles"],
        x_dot=trajectory["derivative_matrix"],
        feature_names=["x", "v"],
    )

    return model

In [33]:
initial_sindy_models = {}

for variability_class in ["RRab", "DSCT", "EW"]:
    model = fit_initial_sindy_model(
        trajectory=sindy_trajectories[variability_class],
        threshold=0.05,
    )

    initial_sindy_models[variability_class] = model

    print("\n" + "=" * 70)
    print(f"MODELO SINDy: {variability_class}")
    print("=" * 70)

    model.print()


MODELO SINDy: RRab
(x)' =  1.000 v
(v)' = -104.465 1 + -57.869 x + -17.876 v +  40.814 x^2 + -27.687 x v +  1.478 v^2 + -75.976 x^3 +  12.376 x^2 v +  0.141 x v^2 + -0.023 v^3

MODELO SINDy: DSCT
(x)' =  1.000 v
(v)' = -1.611 1 + -18.822 x + -0.158 v + -2.115 x^2 + -1.873 x v +  0.146 v^2 + -11.795 x^3 +  0.068 x^2 v + -0.205 x v^2 +  0.001 v^3

MODELO SINDy: EW
(x)' =  1.000 v
(v)' =  212.733 1 + -232.956 x + -1.133 v + -22.190 x^2 +  1.029 x v + -1.467 v^2 + -3.825 x^3 +  0.843 x^2 v + -0.842 x v^2 +  0.001 v^3


In [34]:
evaluation_rows = []

for variability_class, model in initial_sindy_models.items():
    trajectory = sindy_trajectories[variability_class]

    state_matrix = trajectory["state_matrix"]
    true_derivatives = trajectory["derivative_matrix"]

    predicted_derivatives = model.predict(
        state_matrix
    )

    residuals = (
        predicted_derivatives
        - true_derivatives
    )

    rmse_x_equation = float(
        np.sqrt(
            np.mean(residuals[:, 0] ** 2)
        )
    )

    rmse_v_equation = float(
        np.sqrt(
            np.mean(residuals[:, 1] ** 2)
        )
    )

    std_x_derivative = float(
        np.std(true_derivatives[:, 0])
    )

    std_v_derivative = float(
        np.std(true_derivatives[:, 1])
    )

    normalized_rmse_x = (
        rmse_x_equation / std_x_derivative
        if std_x_derivative > 0
        else np.nan
    )

    normalized_rmse_v = (
        rmse_v_equation / std_v_derivative
        if std_v_derivative > 0
        else np.nan
    )

    coefficients = model.coefficients()

    active_first_equation = int(
        np.count_nonzero(
            np.abs(coefficients[0]) > 0
        )
    )

    active_second_equation = int(
        np.count_nonzero(
            np.abs(coefficients[1]) > 0
        )
    )

    evaluation_rows.append(
        {
            "variability_class": variability_class,
            "threshold": 0.05,
            "rmse_x_equation": rmse_x_equation,
            "rmse_v_equation": rmse_v_equation,
            "normalized_rmse_x": normalized_rmse_x,
            "normalized_rmse_v": normalized_rmse_v,
            "active_terms_x_equation": active_first_equation,
            "active_terms_v_equation": active_second_equation,
        }
    )

initial_sindy_evaluation = pd.DataFrame(
    evaluation_rows
)

print(
    initial_sindy_evaluation.to_string(
        index=False
    )
)

variability_class  threshold  rmse_x_equation  rmse_v_equation  normalized_rmse_x  normalized_rmse_v  active_terms_x_equation  active_terms_v_equation
             RRab       0.05     2.386071e-14       214.389436       1.991757e-15           0.619020                        1                       10
             DSCT       0.05     6.624833e-15        25.653781       1.005399e-15           0.487724                        1                       10
               EW       0.05     3.719605e-15        69.880452       2.618354e-16           0.264761                        4                       10


In [35]:
preparation_path = (
    TABLE_DIR
    / "pilot_sindy_preparation.csv"
)

derivative_summary_path = (
    TABLE_DIR
    / "pilot_derivative_summary.csv"
)

evaluation_path = (
    TABLE_DIR
    / "pilot_initial_sindy_evaluation.csv"
)

preparation_summary.to_csv(
    preparation_path,
    index=False,
)

derivative_summary.to_csv(
    derivative_summary_path,
    index=False,
)

initial_sindy_evaluation.to_csv(
    evaluation_path,
    index=False,
)

print("Preparación:", preparation_path.exists())
print("Derivadas:", derivative_summary_path.exists())
print("Evaluación:", evaluation_path.exists())

Preparación: True
Derivadas: True
Evaluación: True


In [36]:
threshold_grid = [
    0.05,
    0.10,
    0.20,
    0.50,
    1.00,
    2.00,
    5.00,
    10.00,
]

coefficient_tolerance = 1e-8


def evaluate_second_order_sindy(
    trajectory,
    thresholds,
    train_fraction=0.70,
):
    state_matrix = trajectory[
        "state_matrix"
    ]

    acceleration = trajectory[
        "second_derivative"
    ]

    number_of_samples = len(
        state_matrix
    )

    split_index = int(
        np.floor(
            train_fraction
            * number_of_samples
        )
    )

    if split_index < 100:
        raise ValueError(
            "Muy pocos datos de entrenamiento."
        )

    if number_of_samples - split_index < 100:
        raise ValueError(
            "Muy pocos datos de validación."
        )

    x_train = state_matrix[
        :split_index
    ]

    y_train = acceleration[
        :split_index
    ].reshape(-1, 1)

    x_validation = state_matrix[
        split_index:
    ]

    y_validation = acceleration[
        split_index:
    ]

    feature_library = (
        ps.PolynomialLibrary(
            degree=3,
            include_bias=True,
        )
    )

    theta_train = (
        feature_library.fit_transform(
            x_train
        )
    )

    theta_validation = (
        feature_library.transform(
            x_validation
        )
    )

    feature_names = (
        feature_library.get_feature_names(
            input_features=["x", "v"]
        )
    )

    validation_scale = float(
        np.std(y_validation)
    )

    if validation_scale <= 0:
        raise ValueError(
            "La aceleración de validación "
            "no tiene variación."
        )

    rows = []
    fitted_models = {}

    for threshold in thresholds:
        optimizer = ps.STLSQ(
            threshold=threshold,
            alpha=1e-5,
            max_iter=100,
            normalize_columns=True,
        )

        optimizer.fit(
            theta_train,
            y_train,
        )

        coefficients = np.asarray(
            optimizer.coef_[0],
            dtype=float,
        )

        acceleration_prediction = (
            theta_validation
            @ coefficients
        )

        residual = (
            acceleration_prediction
            - y_validation
        )

        validation_rmse = float(
            np.sqrt(
                np.mean(residual**2)
            )
        )

        normalized_validation_rmse = (
            validation_rmse
            / validation_scale
        )

        active_mask = (
            np.abs(coefficients)
            > coefficient_tolerance
        )

        active_terms = int(
            active_mask.sum()
        )

        # Penalización ligera por complejidad.
        # El error sigue teniendo mayor peso.
        selection_score = (
            normalized_validation_rmse
            + 0.02 * active_terms
        )

        equation_terms = []

        for feature_name, coefficient in zip(
            feature_names,
            coefficients,
        ):
            if (
                abs(coefficient)
                > coefficient_tolerance
            ):
                equation_terms.append(
                    f"{coefficient:+.6g}*"
                    f"{feature_name}"
                )

        equation_text = (
            " ".join(equation_terms)
            if equation_terms
            else "0"
        )

        rows.append(
            {
                "threshold": threshold,
                "train_points": split_index,
                "validation_points": (
                    number_of_samples
                    - split_index
                ),
                "validation_rmse": (
                    validation_rmse
                ),
                "normalized_validation_rmse": (
                    normalized_validation_rmse
                ),
                "active_terms": active_terms,
                "selection_score": (
                    selection_score
                ),
                "equation": equation_text,
            }
        )

        fitted_models[threshold] = {
            "optimizer": optimizer,
            "coefficients": coefficients,
            "feature_names": feature_names,
            "split_index": split_index,
            "validation_prediction": (
                acceleration_prediction
            ),
            "validation_target": (
                y_validation
            ),
        }

    results = pd.DataFrame(rows)

    results = results.sort_values(
        [
            "selection_score",
            "active_terms",
            "threshold",
        ],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    return results, fitted_models

In [38]:
threshold_results = {}
threshold_models = {}
all_threshold_rows = []

for variability_class in [
    "RRab",
    "DSCT",
    "EW",
]:
    results, fitted_models = (
        evaluate_second_order_sindy(
            trajectory=sindy_trajectories[
                variability_class
            ],
            thresholds=threshold_grid,
            train_fraction=0.70,
        )
    )

    results.insert(
        0,
        "variability_class",
        variability_class,
    )

    threshold_results[
        variability_class
    ] = results

    threshold_models[
        variability_class
    ] = fitted_models

    all_threshold_rows.append(
        results
    )

    print("\n" + "=" * 75)
    print(
        f"RESULTADOS: {variability_class}"
    )
    print("=" * 75)

    print(
        results[
            [
                "threshold",
                "normalized_validation_rmse",
                "active_terms",
                "selection_score",
            ]
        ].to_string(index=False)
    )

ValueError: Assigned definition to axis 1, but array only has 1 axes